# Tracker Analysis Plots

In [1]:
import os
from pathlib import Path
import copy

os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

import pandas as pd
from tgf import TaskGroup, Flag, AbstractFlag, Pipeline
from plotting import PlotPipelineFactory
from concurrent.futures import ThreadPoolExecutor
import warnings

warnings.filterwarnings("ignore")


Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
dataPath = str(Path.cwd()) + r"\data\csv" + "\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\processing_pipeline" + "\\"
plotPath = str(Path.cwd()) + r"\out\plots\tracker_analysis" + "\\"

In [3]:
import pickle

with open(picklePath + 'Processing Pipeline.pickle', 'rb') as f:
    AnalysisPipeline = pickle.load(f)

AnalysisPipeline.print(priority=True, flags=True)

Processing Pipeline
	10 Pre Processing: Pre Processing
		10 Select and order columns
		15 Company ID and UUID
			10 Fill Company ID with None
			20 Fill UUID with None
			30 Replace Company IDs
			40 Replace Company UUIDs
		20 MS Data Processing
			10 Fill MS Data with empty String
			20 Length of MS Data
			30 Continuity Type
			40 Drop MS Data Column
		25 Service Data Processing
			10 Fill Service Data with empty String
			20 Length of Service Data
			30 Samsung Type
			40 Drop Service Data Column
		30 Fill Numeric NA with 0
		40 Fill String NA with None
		50 Broadcast
		60 Datetime conversion
		80 Order DataFrame
		90 Convert object type to string
	20 Dummy Processing: Dummy Processing
		30 Dummies Channel
		40 Dummies AD Type
		50 Dummies Company
		55 Dummies UUID
		70 Dummies PDU Type
		80 Dummies Continuity Type
		90 Dummies SmartTag Type
	30 Labeling: Labeling
		10 Labeling auto: Labeling auto
			Label Apple AirTag and Owner Device: Label Apple AirTag and nearby Owner Device
			

In [4]:
AnalysisPipeline_part_1 = AnalysisPipeline[0]
AnalysisPipeline_part_2 = AnalysisPipeline
del AnalysisPipeline_part_2[0]

In [5]:
with open(picklePath + 'Pre Processing.pickle', 'rb') as f:
    flag_preProcessing = pickle.load(f)

with open(picklePath + 'Modeling.pickle', 'rb') as f:
    flag_modeling = pickle.load(f)

with open(picklePath + 'Dummy Processing.pickle', 'rb') as f:
    flag_dummy = pickle.load(f)

with open(picklePath + 'State lost.pickle', 'rb') as f:
    flag_state_lost = pickle.load(f)

with open(picklePath + 'State unpaired.pickle', 'rb') as f:
    flag_state_unpaired = pickle.load(f)

with open(picklePath + 'State nearby.pickle', 'rb') as f:
    flag_state_nearby = pickle.load(f)

with open(picklePath + 'State searching.pickle', 'rb') as f:
    flag_state_searching = pickle.load(f)

with open(picklePath + 'State online.pickle', 'rb') as f:
    flag_state_online = pickle.load(f)

with open(picklePath + 'State offline.pickle', 'rb') as f:
    flag_state_offline = pickle.load(f)

with open(picklePath + 'States iDevices.pickle', 'rb') as f:
    flag_states_iDevices = pickle.load(f)

with open(picklePath + 'Label other Device.pickle', 'rb') as f:
    flag_label_other_device = pickle.load(f)

In [6]:
def getPlotPipeline(titleSuffix: str, labelColumn: str = 'Label', show: bool = False, savePath: str = plotPath,
                    mode: str = 'analysis') -> TaskGroup:
    return PlotPipelineFactory(titleSuffix=titleSuffix, labelColumn=labelColumn, show=show, savePath=savePath,
                               mode=mode)

In [7]:
def filesToDataFrame(filesDict: dict[str, list[AbstractFlag | str]], filePath: str = dataPath,
                     labelColumn: str = 'Label',
                     dropLabels: list[str] = None) -> pd.DataFrame:
    if dropLabels is None:
        dropLabels = []

    def process_file(file: str, flags: list[AbstractFlag | str]) -> pd.DataFrame:
        myPipeline = Pipeline().setPath(filePath + file).loadData()

        new_dataset = myPipeline.setTask(copy.deepcopy(AnalysisPipeline_part_1)).run()

        parent_flags = []

        for flag in flags:
            if isinstance(flag, AbstractFlag):
                parent_flags.append(flag)

            elif isinstance(flag, str):
                with open(picklePath + flag + ".pickle", 'rb') as flag_file:
                    loaded_flag = pickle.load(flag_file)

                flag_file.close()

                assert isinstance(loaded_flag, AbstractFlag)
                parent_flags.append(loaded_flag)

        processing_flag = Flag("Processing Flag", parents=parent_flags)

        new_dataset_labeled = copy.deepcopy(AnalysisPipeline_part_2).process(new_dataset, flag=processing_flag)

        new_dataset[labelColumn] = new_dataset_labeled[labelColumn]
        new_dataset['File'] = file.split("\\")[-1][:-4]

        return new_dataset

    dfs = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        dfs = list(executor.map(lambda item: process_file(item[0], item[1]), filesDict.items()))

    dataset = pd.concat(dfs)

    for label in dropLabels:
        dataset = dataset[~dataset[labelColumn].str.contains(label)]

    dataset = dataset[~dataset[labelColumn].isin(dropLabels)]

    dataset.reset_index(drop=True, inplace=True)

    return dataset

In [8]:
files = {r"BLE Tracker\Apple Find My\Apple AirTag 2\Apple_AirTag_2_(nearby).csv": [flag_preProcessing,
                                                                                   flag_dummy,
                                                                                   "Label Apple AirTag 2 and nearby Owner Device",
                                                                                   flag_state_nearby],

         r"BLE Tracker\Apple Find My\Apple AirTag 2\Apple_AirTag_2_(lost).csv": [flag_preProcessing,
                                                                                 flag_dummy,
                                                                                 "Label Apple AirTag 2",
                                                                                 flag_state_lost],
         r"BLE Tracker\Apple Find My\Apple AirTag 2\Apple_AirTag_2_(unpaired).csv": [flag_preProcessing,
                                                                                     flag_dummy,
                                                                                     "Label Apple AirTag 2",
                                                                                     flag_state_unpaired],

         r"BLE Tracker\Apple Find My\Apple AirTag\Apple_AirTag_(nearby).csv": [flag_preProcessing,
                                                                               flag_dummy,
                                                                               "Label Apple AirTag and nearby Owner Device",
                                                                               flag_state_nearby],
         r"BLE Tracker\Apple Find My\Apple AirTag\Apple_AirTag_(lost).csv": [flag_preProcessing,
                                                                             flag_dummy,
                                                                             "Label Apple AirTag",
                                                                             flag_state_lost],
         r"BLE Tracker\Apple Find My\Apple AirTag\Apple_AirTag_(unpaired).csv": [flag_preProcessing,
                                                                                 flag_dummy,
                                                                                 "Label Apple AirTag",
                                                                                 flag_state_unpaired],

         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])
name = 'Apple AirTags'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [9]:
files = {r"BLE Tracker\Apple Find My\Chipolo ONE\Chipolo_ONE_(nearby).csv": [flag_preProcessing,
                                                                             flag_dummy,
                                                                             "Label Chipolo ONE and nearby Owner Device",
                                                                             flag_state_nearby],
         r"BLE Tracker\Apple Find My\Chipolo ONE\Chipolo_ONE_(lost).csv": [flag_preProcessing, flag_dummy,
                                                                           "Label Chipolo ONE",
                                                                           flag_state_lost],
         r"BLE Tracker\Apple Find My\Chipolo ONE\Chipolo_ONE_(unpaired).csv": [flag_preProcessing,
                                                                               flag_dummy,
                                                                               "Label Chipolo ONE",
                                                                               flag_state_unpaired],

         r"BLE Tracker\Apple Find My\4Smarts SkyTag\4Smarts_SkyTag_(nearby).csv": [flag_preProcessing,
                                                                                   flag_dummy,
                                                                                   "Label 4Smarts SkyTag and nearby Owner Device",
                                                                                   flag_state_nearby],
         r"BLE Tracker\Apple Find My\4Smarts SkyTag\4Smarts_SkyTag_(lost).csv": [flag_preProcessing,
                                                                                 flag_dummy,
                                                                                 "Label 4Smarts SkyTag",
                                                                                 flag_state_lost],
         r"BLE Tracker\Apple Find My\4Smarts SkyTag\4Smarts_SkyTag_(unpaired).csv": [flag_preProcessing,
                                                                                     flag_dummy,
                                                                                     "Label 4Smarts SkyTag",
                                                                                     flag_state_unpaired]

         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])
name = 'Apple Find My Tracker'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [10]:
files = {r"BLE Tracker\Apple Find My\KeySmart SmartCard\KeySmart_SmartCard_(nearby).csv": [flag_preProcessing,
                                                                                           flag_dummy,
                                                                                           "Label KeySmart SmartCard and nearby Owner Device",
                                                                                           flag_state_nearby],
         r"BLE Tracker\Apple Find My\KeySmart SmartCard\KeySmart_SmartCard_(lost).csv": [flag_preProcessing,
                                                                                         flag_dummy,
                                                                                         "Label KeySmart SmartCard",
                                                                                         flag_state_lost],
         r"BLE Tracker\Apple Find My\KeySmart SmartCard\KeySmart_SmartCard_(unpaired).csv": [flag_preProcessing,
                                                                                             flag_dummy,
                                                                                             "Label KeySmart SmartCard",
                                                                                             flag_state_unpaired],



        r"BLE Tracker\Apple Find My\Chipolo CARD\Chipolo_CARD_(nearby)_(Apple).csv": [flag_preProcessing,
                                                                                       flag_dummy,
                                                                                       "Label Chipolo CARD [Apple] and nearby Owner Device",
                                                                                       flag_state_nearby],
         r"BLE Tracker\Apple Find My\Chipolo CARD\Chipolo_CARD_(lost)_(Apple).csv": [flag_preProcessing, flag_dummy,
                                                                                     "Label Chipolo CARD [Apple]",
                                                                                     flag_state_lost],
         r"BLE Tracker\Apple Find My\Chipolo CARD\Chipolo_CARD_(unpaired).csv": [flag_preProcessing,
                                                                                 flag_dummy,
                                                                                 "Label Chipolo CARD [Apple]",
                                                                                 flag_state_unpaired],

         r"BLE Tracker\Apple Find My\4Smarts SkyTag Card\4Smarts_SkyTag_Card_(nearby).csv": [flag_preProcessing,
                                                                                             flag_dummy,
                                                                                             "Label 4Smarts SkyTag Card and nearby Owner Device",
                                                                                             flag_state_nearby],
         r"BLE Tracker\Apple Find My\4Smarts SkyTag Card\4Smarts_SkyTag_Card_(lost).csv": [flag_preProcessing,
                                                                                           flag_dummy,
                                                                                           "Label 4Smarts SkyTag Card",
                                                                                           flag_state_lost],
         r"BLE Tracker\Apple Find My\4Smarts SkyTag Card\4Smarts_SkyTag_Card_(unpaired).csv": [flag_preProcessing,
                                                                                               flag_dummy,
                                                                                               "Label 4Smarts SkyTag Card",
                                                                                               flag_state_unpaired],



         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])
name = 'Apple Find My Tracker Cards'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [11]:
files = {r"BLE Tracker\Google Find My\Motorola MotoTag\Motorola_MotoTag_(nearby).csv": [flag_preProcessing,
                                                                                        flag_dummy,
                                                                                        "Label Motorola MotoTag and nearby Owner Device",
                                                                                        flag_state_nearby],

         r"BLE Tracker\Google Find My\Motorola MotoTag\Motorola_MotoTag_(lost).csv": [flag_preProcessing,
                                                                                      flag_dummy,
                                                                                      "Label Motorola MotoTag",
                                                                                      flag_state_lost],

         r"BLE Tracker\Google Find My\Motorola MotoTag\Motorola_MotoTag_(unpaired).csv": [flag_preProcessing,
                                                                                          flag_dummy,
                                                                                          "Label Motorola MotoTag",
                                                                                          flag_state_unpaired],

         r"BLE Tracker\Google Find My\Lifemate LifeTag\Lifemate_LifeTag_(nearby).csv": [flag_preProcessing,
                                                                                        flag_dummy,
                                                                                        "Label Lifemate LifeTag and nearby Owner Device",
                                                                                        flag_state_nearby],

         r"BLE Tracker\Google Find My\Lifemate LifeTag\Lifemate_LifeTag_(lost).csv": [flag_preProcessing,
                                                                                      flag_dummy,
                                                                                      "Label Lifemate LifeTag",
                                                                                      flag_state_lost],

         r"BLE Tracker\Google Find My\Lifemate LifeTag\Lifemate_LifeTag_(unpaired).csv": [flag_preProcessing,
                                                                                          flag_dummy,
                                                                                          "Label Lifemate LifeTag",
                                                                                          flag_state_unpaired],

         r"BLE Tracker\Google Find My\Hama MGF\Hama_MGF_(nearby).csv": [flag_preProcessing,
                                                                        flag_dummy,
                                                                        "Label Hama MGF and nearby Owner Device",
                                                                        flag_state_nearby],

         r"BLE Tracker\Google Find My\Hama MGF\Hama_MGF_(lost).csv": [flag_preProcessing,
                                                                      flag_dummy,
                                                                      "Label Hama MGF",
                                                                      flag_state_lost],

         r"BLE Tracker\Google Find My\Hama MGF\Hama_MGF_(unpaired).csv": [flag_preProcessing,
                                                                          flag_dummy,
                                                                          "Label Hama MGF",
                                                                          flag_state_unpaired],

         r"BLE Tracker\Google Find My\Chipolo CARD\Chipolo_CARD_(nearby)_(Google).csv": [flag_preProcessing,
                                                                                         flag_dummy,
                                                                                         "Label Chipolo CARD [Google] and nearby Owner Device",
                                                                                         flag_state_nearby],

         r"BLE Tracker\Google Find My\Chipolo CARD\Chipolo_CARD_(lost)_(Google).csv": [flag_preProcessing, flag_dummy,
                                                                                       "Label Chipolo CARD [Google]",
                                                                                       flag_state_lost],

         r"BLE Tracker\Google Find My\Chipolo CARD\Chipolo_CARD_(unpaired).csv": [flag_preProcessing,
                                                                                  flag_dummy,
                                                                                  "Label Chipolo CARD [Google]",
                                                                                  flag_state_unpaired],

         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])
name = 'Google Find My Tracker'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [12]:
files = {r"BLE Tracker\Samsung SmartThings\Samsung SmartTag\Samsung_SmartTag_(nearby).csv": [flag_preProcessing,
                                                                                             flag_dummy,
                                                                                             "Label Samsung SmartTag and nearby Owner Device",
                                                                                             flag_state_nearby],
         r"BLE Tracker\Samsung SmartThings\Samsung SmartTag\Samsung_SmartTag_(lost).csv": [flag_preProcessing,
                                                                                           flag_dummy,
                                                                                           "Label Samsung SmartTag",
                                                                                           flag_state_lost],
         r"BLE Tracker\Samsung SmartThings\Samsung SmartTag\Samsung_SmartTag_(unpaired).csv": [flag_preProcessing,
                                                                                               flag_dummy,
                                                                                               "Label Samsung SmartTag",
                                                                                               flag_state_unpaired],
         r"BLE Tracker\Samsung SmartThings\Samsung SmartTag\Samsung_SmartTag_(searching).csv": [flag_preProcessing,
                                                                                                flag_dummy,
                                                                                                "Label Samsung SmartTag",
                                                                                                flag_state_searching]
         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])

name = 'Samsung SmartThings Tracker'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [13]:
files = {r"BLE Tracker\Tile\Tile Mate\Tile_Mate_(nearby).csv": [flag_preProcessing,
                                                                flag_dummy,
                                                                "Label Tile Mate and nearby Owner Device",
                                                                flag_state_nearby],
         r"BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost).csv": [flag_preProcessing,
                                                              flag_dummy,
                                                              "Label Tile Mate",
                                                              flag_state_lost],
         r"BLE Tracker\Tile\Tile Mate\Tile_Mate_(unpaired).csv": [flag_preProcessing,
                                                                  flag_dummy,
                                                                  "Label Tile Mate",
                                                                  flag_state_unpaired],
         r"BLE Tracker\Tile\Tile Mate\Tile_Mate_(searching).csv": [flag_preProcessing,
                                                                   flag_dummy,
                                                                   "Label Tile Mate",
                                                                   flag_state_searching],

         r"BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby).csv": [flag_preProcessing,
                                                                flag_dummy,
                                                                "Label Tile Slim and nearby Owner Device",
                                                                flag_state_nearby],
         r"BLE Tracker\Tile\Tile Slim\Tile_Slim_(lost).csv": [flag_preProcessing,
                                                              flag_dummy,
                                                              "Label Tile Slim",
                                                              flag_state_lost],
         r"BLE Tracker\Tile\Tile Slim\Tile_Slim_(unpaired).csv": [flag_preProcessing,
                                                                  flag_dummy,
                                                                  "Label Tile Slim",
                                                                  flag_state_unpaired],
         }

data = filesToDataFrame(files, dropLabels=['Owner Device'])

name = 'Tile Tracker'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [14]:
files = {r"Apple iDevices\Apple iPhone\Apple_iPhone_(online).csv": [flag_preProcessing,
                                                                    flag_dummy,
                                                                    "Label Apple iPhone",
                                                                    flag_state_online],
         r"Apple iDevices\Apple iPhone\Apple_iPhone_(offline).csv": [flag_preProcessing,
                                                                     flag_dummy,
                                                                     "Label Apple iPhone",
                                                                     flag_state_offline],
         }

data = filesToDataFrame(files, dropLabels=['Apple iPhone (offline)', 'Apple iPhone (online)'])

name = 'Apple iPhone'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [15]:
files = {r"Apple iDevices\Apple iPad\Apple_iPad_(online).csv": [flag_preProcessing,
                                                                flag_dummy,
                                                                "Label Apple iPad",
                                                                flag_state_online],
         r"Apple iDevices\Apple iPad\Apple_iPad_(offline).csv": [flag_preProcessing,
                                                                 flag_dummy,
                                                                 "Label Apple iPad",
                                                                 flag_state_offline],
         }

data = filesToDataFrame(files, dropLabels=['Apple iPad (offline)', 'Apple iPad (online)', 'Apple iPad CT 09 (offline)'])

name = 'Apple iPad'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [16]:
files = {r"Apple iDevices\Apple MacBook\Apple_MacBook_(online).csv": [flag_preProcessing,
                                                                      flag_dummy,
                                                                      "Label Apple MacBook",
                                                                      flag_state_online],
         r"Apple iDevices\Apple MacBook\Apple_MacBook_(offline).csv": [flag_preProcessing,
                                                                       flag_dummy,
                                                                       "Label Apple MacBook",
                                                                       flag_state_offline],
         }

data = filesToDataFrame(files, dropLabels=['Apple MacBook (offline)', 'Apple MacBook (online)'])

name = 'Apple MacBook'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [17]:
files = {r"Apple iDevices\Apple AirPod\Apple_AirPod_(nearby).csv": [flag_preProcessing,
                                                                    flag_dummy,
                                                                    "Label Apple AirPod and nearby Owner Device",
                                                                    flag_states_iDevices,
                                                                    flag_state_nearby],
         r"Apple iDevices\Apple AirPod\Apple_AirPod_(lost).csv": [flag_preProcessing,
                                                                  flag_dummy,
                                                                  "Label Apple AirPod",
                                                                  flag_states_iDevices,
                                                                  flag_state_lost],
         }

data = filesToDataFrame(files, dropLabels=['Apple AirPod (lost)', 'Apple AirPod (nearby)', 'Owner Device',
                                           'Apple AirPod CT 09 (nearby)'])

name = 'Apple AirPod'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)

In [18]:
files = {r"other Device\other Device.csv": [flag_preProcessing, flag_dummy, 'Label other Device']}

data = filesToDataFrame(files)

name = 'other Device'
plotPipeline = getPlotPipeline(name, savePath=plotPath + name + "\\")
data = plotPipeline.execute(data)